# Build WLASL LMDB (ASL Signs-compatible)

WLASL landmarks (output of `preprocess-wlasl-mediapipe`, legacy MediaPipe Holistic) for the
signs shared with ASL Signs (233 WLASL glosses → 206 signs), written with the same
preprocessing and key scheme as `asl-is-lmdb`, plus a per-clip aspect correction
(WLASL is landscape, ASL Signs portrait ≈ 3:4). Logic lives in the repo:
`research/models/cnn_transformer/data/build_wlasl_lmdb.py`. CPU only.

Output (`/kaggle/working/wlasl-lmdb`): `wlasl.lmdb.mdb`, `train.csv`, `sign_to_prediction_index_map.json`.

In [ ]:
!pip install -q lmdb
!git clone -q --depth 1 https://github.com/shravnchandr/Sign-Language-Translation.git /tmp/repo
!git -C /tmp/repo log --oneline -1

In [ ]:
import glob, os

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    print(f"{pattern}: {hits[:3]}")
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

wlasl_dir = os.path.dirname(find("wlasl_landmarks/train.csv"))
wlasl_json = find("WLASL_v0.3.json")
sign_map = find("asl-signs/sign_to_prediction_index_map.json")  # not WLASL's 2,000-sign map
import json
assert len(json.load(open(sign_map))) == 250, sign_map
n = len(glob.glob(f"{wlasl_dir}/train_landmark_files/*.parquet"))
print(f"{n} WLASL parquets in {wlasl_dir}")

In [ ]:
import subprocess, sys
cmd = [sys.executable, "-m", "cnn_transformer.data.build_wlasl_lmdb",
       "--wlasl-dir", wlasl_dir, "--wlasl-json", wlasl_json, "--sign-map", sign_map,
       "--out-dir", "/kaggle/working/wlasl-lmdb", "--workers", str(os.cpu_count())]
env = {**os.environ, "PYTHONPATH": "/tmp/repo/research/models"}
subprocess.run(cmd, env=env, check=True)

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/wlasl-lmdb/train.csv")
print(df.describe(include="all").T[["count", "unique", "mean", "50%"]].to_string())
print("\nclips per sign:", df["sign"].value_counts().describe().round(1).to_dict())
print("synonym-mapped glosses:", df.loc[df["synonym"], ["wlasl_gloss", "sign"]].drop_duplicates().values.tolist())
!ls -la /kaggle/working/wlasl-lmdb